# NeSy Runtime Monitoring — experiments on Colab

Runs all timing + capability experiments and regenerates every figure.
Each experiment script writes its CSV(s) to `results/` and renders its PNG(s)
via `experiments/plots.py`; the final **Plots** cell re-renders everything from
the CSVs. See `results/README.md` for the full experiment × plot matrix.

**Device:** every experiment auto-selects `cuda` when a GPU runtime is active,
else `cpu` (the symbolic DFA always runs on CPU — it is a pure-Python dict walk).
For a CPU-vs-GPU comparison, run this notebook once on a **CPU** runtime and once
on a **GPU** runtime and keep the two `results/` folders as described in
`results/README.md`.

## Setup

In [ ]:
!apt-get -qq install -y mona
!git clone https://github.com/matteomancanelli/nesy_runtime_monitoring.git
%cd nesy_runtime_monitoring
!pip install -q -e .

### (optional) Persist results to Google Drive

Redirects the repo's `results/` to a Drive folder so CSVs/PNGs survive a VM
reset. Skip this cell to keep results only on the ephemeral VM.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
dst = '/content/drive/MyDrive/nesy_results'
os.makedirs(dst, exist_ok=True)

# Redirect the repo's results/ to Drive so CSVs survive a VM reset.
!rm -rf /content/nesy_runtime_monitoring/results
!ln -s {dst} /content/nesy_runtime_monitoring/results

## Sanity check

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name() if torch.cuda.is_available() else "(CPU runtime)")
!python -c "from ltlf2dfa.ltlf2dfa import to_dfa; print('mona OK')"

## Run experiments

Each script is resumable (incremental CSV) and renders its own figure(s) at the
end. Timing sweeps (exp1/2/3/5/6) can be slow — exp2/exp3 are the heaviest.

Every timing experiment runs the full monitor set, including **both RuleRunner
paradigms**: the original (flat + structured) and the corrected **progression**
RuleRunner (flat + structured). On exp2 the progression pair shares DeepDFA-dense's
`2^|AP|` alphabet wall and is capped at n≤16 leaves.

The two **capability / finding** experiments are cheaper than the timing sweeps:
`exp_uncertainty` (Capability A — accuracy + calibration vs noise) and
`exp7_richer_family` (Phase 3.3 — the non-read-once soft-divergence curve, which
needs no GPU, plus the state-blowup neutrality panel, whose timing uses the GPU
when present).

In [ ]:
!python experiments/exp1_single_trace.py       # per-cell cost vs trace length

In [ ]:
!python experiments/exp2_formula_complexity.py # per-cell cost vs formula size + memory wall

In [ ]:
!python experiments/exp3_batch_size.py         # throughput vs batch size (cross-trace)

In [ ]:
!python experiments/exp5_depth_microbench.py   # within-step cost vs nested-X depth

In [ ]:
!python experiments/exp6_state_scaling.py      # per-cell cost vs automaton size |Q|

In [ ]:
!python experiments/exp_uncertainty.py         # Capability A: accuracy + calibration vs noise

In [ ]:
!python experiments/exp7_richer_family.py      # Phase 3.3: soft-divergence curve + state-blowup neutrality

## Generate all plots

Re-renders every figure from the CSVs currently in `results/` (idempotent — the
experiment scripts already produced these, this regenerates them in one shot).

In [ ]:
!python experiments/plots.py

## Cost of correctness — progression vs original RuleRunner

Prints the corrected/original per-cell-time **ratio** (the paradigm-2 paper
number) on the flat IJCNN family, where the *original* RuleRunner is also
correct so the ratio isolates the encoding's throughput cost, not the verdict
fix (>1 = the correctness fix is slower). Also writes `results/correctness_cost.png`.
Needs `results/exp2_formula_complexity.csv` (run exp2 above first).

In [ ]:
!python experiments/plots.py correctness

## Download results

In [ ]:
from google.colab import files
!cd results && zip -r /content/results.zip . && echo done
files.download('/content/results.zip')